# HGRIA - Hand Gesture Recognition for Interactive Applications
## Launch Notebook

```
┌─────────────────────────────────────────────────────────────┐
│                    ARCHITECTURE                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   Browser (Frontend)     ngrok Tunnel      Colab Backend   │
│   ┌──────────────┐      ┌──────────┐     ┌──────────────┐   │
│   │  GitHub Pages │ ←─── │  HTTPS   │ ←── │  Flask +     │   │
│   │  / Vercel    │      │  Tunnel  │     │  MediaPipe   │   │
│   └──────────────┘      └──────────┘     └──────────────┘   │
│         │                                          │        │
│         │           Google Drive                    │        │
│         └──────────────┬───────────────────────────┘        │
│                        │ logs/                             │
└─────────────────────────────────────────────────────────────┘
```

### Prerequisites
- Google Account with Google Drive access
- ngrok account (free tier works)
- WebRTC-compatible browser (Chrome, Edge, Firefox)

### How it works
1. Backend runs Flask server on Colab with MediaPipe
2. ngrok creates HTTPS tunnel to expose backend
3. Frontend connects via WebSocket and sends webcam frames
4. Backend processes frames and sends gesture commands back

In [ ]:
!pip install --no-cache-dir \
    "numpy==1.26.4" \
    "tensorflow==2.18.0" \
    "protobuf==4.25.3" \
    "mediapipe==0.10.21" \
    "opencv-contrib-python==4.11.0.86"

In [ ]:
# Step 1: Mount Google Drive and verify project files
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

SOURCE_PATH = '/content/drive/MyDrive/HGRIA'
REQUIREMENTS_FILE = os.path.join(SOURCE_PATH, 'requirements.txt')

if not os.path.exists(REQUIREMENTS_FILE):
    raise FileNotFoundError(
        f"requirements.txt not found at {REQUIREMENTS_FILE}.\n"
        "Please ensure HGRIA project is saved in your Google Drive at:\n"
        "  /content/drive/MyDrive/HGRIA/"
    )

print(f"✓ Project files found at {SOURCE_PATH}")

Mounted at /content/drive
✓ Project files found at /content/drive/MyDrive/HGRIA


In [ ]:
# !pip uninstall -y mediapipe tensorflow tensorflow-cpu protobuf numpy jax jaxlib

In [ ]:
import numpy as np
import google.protobuf
import mediapipe as mp
import tensorflow as tf

print("NumPy:", np.__version__)
print("Protobuf:", google.protobuf.__version__)
print("MediaPipe:", mp.__version__)
print("TensorFlow:", tf.__version__)
print("MediaPipe OK")
print("TensorFlow OK")

NumPy: 1.26.4
Protobuf: 4.25.3
MediaPipe: 0.10.21
TensorFlow: 2.18.0
MediaPipe OK
TensorFlow OK


In [ ]:
# Step 2: Copy project to Colab and install dependencies
import shutil
import subprocess
import sys

DEST = '/content/HGRIA'

# Copy only if destination doesn't exist or user wants to refresh
if os.path.exists(DEST):
    print(f"Project already exists at {DEST}")
else:
    shutil.copytree(SOURCE_PATH, DEST)
    print(f"✓ Copied project to {DEST}")

# Add to Python path
sys.path.insert(0, DEST)
os.chdir(DEST)

# Install dependencies
result = subprocess.run(
    ['pip', 'install', '-q', '-r', 'requirements.txt'],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        f"Failed to install dependencies:\n{result.stderr}"
    )

print("✓ Dependencies installed successfully")

✓ Copied project to /content/HGRIA
✓ Dependencies installed successfully


In [ ]:
# Step 3: Configure ngrok authentication
import getpass

# Install pyngrok if not present
subprocess.run(['pip', 'install', '-q', 'pyngrok'], capture_output=True)
from pyngrok import ngrok

# Get ngrok auth token (optional but recommended)
print("Enter your ngrok authtoken (from https://dashboard.ngrok.com/auth)")
print("Press Enter to skip (anonymous tunnel may disconnect)")

authtoken = getpass.getpass(prompt='Authtoken: ')

if authtoken:
    ngrok.set_auth_token(authtoken)
    print("✓ ngrok authenticated")
else:
    print("⚠ Anonymous tunnel - connection may be unstable")

Enter your ngrok authtoken (from https://dashboard.ngrok.com/auth)
Press Enter to skip (anonymous tunnel may disconnect)
Authtoken: ··········
✓ ngrok authenticated


In [ ]:
# Step 4: Start ngrok tunnel and display connection info
import json

# Start ngrok tunnel
tunnel = ngrok.connect(5000, "http")

# Convert HTTP URL to HTTPS
ngrok_url = tunnel.public_url.replace('http://', 'https://')

print("=" * 60)
print("🔗 NGROK TUNNEL READY")
print("=" * 60)
print(f"\nBackend URL: {ngrok_url}")

# Generate frontend integration snippet
frontend_script = f'<script>window.HGRIA_BACKEND_URL="{ngrok_url}";<\/script>'
print(f"\nPaste this in your frontend HTML (before Socket.IO loads):")
print(f"\n{frontend_script}")

# Generate frontend URL (assuming GitHub Pages or Vercel)
frontend_base = "https://qtannguyen-researcher.github.io/HGRIA"
frontend_url = f"{frontend_base}?server={ngrok_url}"

print(f"\nOr open Frontend directly with:")
print(f"\n{frontend_url}")
print("\n" + "=" * 60)

<>:16: SyntaxWarning: invalid escape sequence '\/'
<>:16: SyntaxWarning: invalid escape sequence '\/'
/tmp/ipykernel_1455/908728183.py:16: SyntaxWarning: invalid escape sequence '\/'
  frontend_script = f'<script>window.HGRIA_BACKEND_URL="{ngrok_url}";<\/script>'


🔗 NGROK TUNNEL READY

Backend URL: https://stylized-stark-proofing.ngrok-free.dev

Paste this in your frontend HTML (before Socket.IO loads):

<script>window.HGRIA_BACKEND_URL="https://stylized-stark-proofing.ngrok-free.dev";<\/script>

Or open Frontend directly with:

https://qtannguyen-researcher.github.io/HGRIA?server=https://stylized-stark-proofing.ngrok-free.dev



In [ ]:
# Step 5: Configure and patch config for Colab mode
import json

config_path = os.path.join(DEST, 'config', 'config.json')

# Load existing config
with open(config_path, 'r') as f:
    config = json.load(f)

# Apply Colab-specific patches
config['camera']['colab_mode'] = True
config['server']['cors_origins'] = '*'
config['logging']['log_to_file'] = True
config['logging']['log_file_path'] = '/content/drive/MyDrive/HGRIA/logs/'

# Write patched config back
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("✓ Config patched for Colab mode:")
print(f"  - colab_mode: {config['camera']['colab_mode']}")
print(f"  - cors_origins: {config['server']['cors_origins']}")
print(f"  - log_to_file: {config['logging']['log_to_file']}")
print(f"  - log_file_path: {config['logging']['log_file_path']}")

✓ Config patched for Colab mode:
  - colab_mode: True
  - cors_origins: *
  - log_to_file: True
  - log_file_path: /content/drive/MyDrive/HGRIA/logs/


In [ ]:
# Step 6: Start the HGRIA Server (blocking)
# This cell will keep running until interrupted
from backend.main import SystemOrchestrator

print("Starting HGRIA Backend Server...")
print(f"Server running at: {ngrok_url}")
print("\nPress Stop button to terminate the server")
print("-" * 40)

# Start the orchestrator (blocking)
orchestrator = SystemOrchestrator(config_path)
orchestrator.start()

Starting HGRIA Backend Server...
Server running at: https://stylized-stark-proofing.ngrok-free.dev

Press Stop button to terminate the server
----------------------------------------
{"timestamp": "2026-08-16T11:25:45.100+00:00", "level": "INFO", "module": "orchestrator", "event": "startup_begin"}
{"timestamp": "2026-08-16T11:25:45.101+00:00", "level": "INFO", "module": "orchestrator", "event": "config_loaded"}
{"timestamp": "2026-08-16T11:25:45.106+00:00", "level": "INFO", "module": "orchestrator", "event": "mediapipe_init_start"}
{"timestamp": "2026-08-16T11:25:45.212+00:00", "level": "INFO", "module": "hand_detector", "event": "mediapipe_warmup_complete", "warmup_ms": 61.43}
{"timestamp": "2026-08-16T11:25:45.212+00:00", "level": "INFO", "module": "orchestrator", "event": "mediapipe_ready"}
{"timestamp": "2026-08-16T11:25:45.212+00:00", "level": "INFO", "module": "orchestrator", "event": "camera_ready"}
{"timestamp": "2026-08-16T11:25:46.135+00:00", "level": "INFO", "module": "orche


Public URL: https://stylized-stark-proofing.ngrok-free.dev
{"timestamp": "2026-08-16T11:25:46.398+00:00", "level": "INFO", "module": "orchestrator", "event": "server_starting"}
 * Serving Flask app 'backend.app'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


{"timestamp": "2026-08-16T11:26:00.901+00:00", "level": "INFO", "module": "state_manager", "event": "state_transition", "old": "Idle", "new": "Searching", "transition_event": "client_connected"}
{"timestamp": "2026-08-16T11:30:24.558+00:00", "level": "INFO", "module": "state_manager", "event": "state_transition", "old": "Searching", "new": "Disconnected", "transition_event": "client_disconnected"}


INFO:werkzeug:2001:ee0:4f8c:400:7126:1a1c:421:1083 - - [16/Aug/2026 11:30:24] "GET /socket.io/?EIO=4&transport=websocket HTTP/1.1" 500 -
ERROR:werkzeug:Error on request:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/serving.py", line 371, in run_wsgi
    execute(self.server.app)
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/serving.py", line 337, in execute
    write(b"")
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/serving.py", line 262, in write
    assert status_set is not None, "write() before start_response"
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: write() before start_response


{"timestamp": "2026-08-16T11:30:25.425+00:00", "level": "INFO", "module": "state_manager", "event": "state_transition", "old": "Disconnected", "new": "Searching", "transition_event": "client_connected"}


Streaming output truncated to the last 5000 lines.
INFO:werkzeug:2001:ee0:4f8c:400:7126:1a1c:421:1083 - - [16/Aug/2026 11:35:31] "POST /api/frame HTTP/1.1" 400 -
INFO:werkzeug:2001:ee0:4f8c:400:7126:1a1c:421:1083 - - [16/Aug/2026 11:35:31] "POST /api/frame HTTP/1.1" 400 -
INFO:werkzeug:2001:ee0:4f8c:400:7126:1a1c:421:1083 - - [16/Aug/2026 11:35:31] "POST /api/frame HTTP/1.1" 400 -
INFO:werkzeug:2001:ee0:4f8c:400:7126:1a1c:421:1083 - - [16/Aug/2026 11:35:31] "POST /api/frame HTTP/1.1" 400 -
INFO:werkzeug:2001:ee0:4f8c:400:7126:1a1c:421:1083 - - [16/Aug/2026 11:35:31] "POST /api/frame HTTP/1.1" 400 -
INFO:werkzeug:2001:ee0:4f8c:400:7126:1a1c:421:1083 - - [16/Aug/2026 11:35:31] "POST /api/frame HTTP/1.1" 400 -
INFO:werkzeug:2001:ee0:4f8c:400:7126:1a1c:421:1083 - - [16/Aug/2026 11:35:31] "POST /api/frame HTTP/1.1" 400 -
INFO:werkzeug:2001:ee0:4f8c:400:7126:1a1c:421:1083 - - [16/Aug/2026 11:35:31] "POST /api/frame HTTP/1.1" 400 -
INFO:werkzeug:2001:ee0:4f8c:400:7126:1a1c:421:1083 - - [16/Au

{"timestamp": "2026-08-16T11:38:04.033+00:00", "level": "INFO", "module": "orchestrator", "event": "shutdown_begin"}
{"timestamp": "2026-08-16T11:38:04.033+00:00", "level": "INFO", "module": "pipeline_runner", "event": "pipeline_stopped"}


SystemExit: 0

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
INFO:werkzeug:2001:ee0:4f8c:400:7126:1a1c:421:1083 - - [16/Aug/2026 11:38:04] "GET /socket.io/?EIO=4&transport=websocket HTTP/1.1" 500 -


## Post-Launch Instructions

### Accessing the Frontend

After the server starts, open your browser and navigate to:

```
https://your-username.github.io/HGRIA/?server=<NGROK_URL>
```

Or paste the `window.HGRIA_BACKEND_URL` script into the frontend HTML before Socket.IO.

### If ngrok URL Changes

1. Stop the server (interrupt the cell above)
2. Re-run cells 4, 5, and 6 in sequence
3. Update the frontend with the new URL

### Troubleshooting

| Issue | Solution |
|-------|----------|
| Colab session timeout | Re-run cell 6 (server restart is automatic) |
| ngrok URL changed | Re-run cells 4, 5, 6 and update frontend |
| Webcam denied | Use keyboard fallback (Arrow keys, Space, P, S) |
| High latency | Check Colab GPU availability |

### Keyboard Controls (Fallback)
- Arrow Keys: Move
- Space: Jump
- P: Pause
- S: Speed Boost
- Enter: Confirm